# Value-for-Money QLoRA — Fine-Tuning Notebook (Colab, GPU required)

Fine-tunes an open-source model to predict a wine's value-for-money (VFM)
score, 0-99, from its tasting note and metadata.

**Requires a CUDA GPU** — connect a T4 (free tier, LITE_MODE) or A100
(paid, full dataset) via Runtime > Change runtime type before running.

**Setup:** upload the `utils/` folder to this Colab's file browser
(or `git clone` your repo) before running the import cell.

| Part | Content |
|---|---|
| 1 | Exploration: quantization memory footprint, LoRA parameter counting |
| 2 | Tokenizer sanity check: confirm no truncation needed for our data |
| 3 | Training: QLoRA fine-tune via SFTTrainer |
| 4 | Testing: load the trained adapter, evaluate on the held-out test set |


## Part 1 — Exploration: Quantization and LoRA sizing

Two things worth building intuition for before training: how much memory
quantization saves, and how many trainable parameters LoRA actually adds
(a small fraction of the base model — that's the whole point).

**Note on Colab sessions:** loading the same base model at multiple
precisions in one session can exhaust GPU memory. If you hit a CUDA OOM
error, use Runtime > Restart session between the three loads below.


In [ ]:
!pip install -q --upgrade bitsandbytes trl peft


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

BASE_MODEL: str = "meta-llama/Llama-3.2-3B"


In [ ]:
# ── Load at full precision — establishes the memory baseline ──────────────────
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
print(f"Full precision memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")


In [ ]:
# ── Restart session (Runtime > Restart session), then load at 8-bit ──────────
quant_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config_8bit, device_map="auto",
)
print(f"8-bit memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")


In [ ]:
# ── Restart session again, then load at 4-bit — this is what training uses ───
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config_4bit, device_map="auto",
)
print(f"4-bit memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")
# Expect roughly a 4x reduction vs full precision — this is what makes
# fine-tuning a 3B model feasible on a free T4 GPU.


In [ ]:
# ── LoRA parameter counting — how much does the adapter actually add? ─────────
# Llama 3.2 3B dims: hidden=3072, attention heads project to 3072 (q/o) or
# 1024 (k/v, grouped-query attention), 28 transformer layers.
# Each target module gets 2 small matrices (A, B) of rank r; their product
# approximates a full-size weight UPDATE without ever storing one.

def count_lora_params(r: int, include_mlp: bool) -> tuple[int, float]:
    """Count LoRA adapter parameters for a given rank and layer scope."""
    lora_q_proj = 3072 * r + 3072 * r
    lora_k_proj = 3072 * r + 1024 * r
    lora_v_proj = 3072 * r + 1024 * r
    lora_o_proj = 3072 * r + 3072 * r
    layer_params = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

    if include_mlp:
        lora_gate_proj = 3072 * r + 8192 * r
        lora_up_proj = 3072 * r + 8192 * r
        lora_down_proj = 3072 * r + 8192 * r
        layer_params += lora_gate_proj + lora_up_proj + lora_down_proj

    total_params = layer_params * 28  # 28 transformer layers
    size_mb = (total_params * 4) / 1_000_000  # 4 bytes/param at fp32 storage
    return total_params, size_mb

for label, r, include_mlp in [
    ("Lite (attention only, r=32)", 32, False),
    ("Full (attention + MLP, r=256)", 256, True),
]:
    params, size_mb = count_lora_params(r, include_mlp)
    print(f"{label}: {params:,} params, {size_mb:,.1f} MB")
# Compare against the ~3 BILLION base model params — LoRA trains a tiny
# fraction of that, which is exactly why it fits on modest hardware.


## Part 2 — Tokenizer Sanity Check

Wine summaries are a bounded, templated format (a curated tasting note,
100-2,000 chars, plus five short fixed metadata lines) — nothing like
the highly variable listings this technique was originally built for.
This check confirms a generous cutoff needs no truncation logic to fire,
before committing to a training config.


In [ ]:
import matplotlib.pyplot as plt
from utils.items import Wine
from utils.vfm import apply_vfm

# Load the curated wines (produced and pushed to the Hub in the baseline
# ladder notebook) and re-derive VFM — apply_vfm is a pure function of
# (points, price), so this is safe to re-run.
HUB_DATASET_NAME: str = "your-username/wine-vfm-curated"
train, val, test = Wine.from_hub(HUB_DATASET_NAME)
for split in (train, val, test):
    apply_vfm(split)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

token_counts = [w.count_tokens(tokenizer) for w in train]
plt.figure(figsize=(15, 5))
plt.title(f"Summary tokens: avg {sum(token_counts)/len(token_counts):.1f}, max {max(token_counts)}")
plt.hist(token_counts, bins=40, color="#4A6FA5")
plt.xlabel("Tokens"); plt.ylabel("Count")
plt.show()


In [ ]:
CUTOFF: int = 200  # generous; adjust upward only if the histogram above shows truncation risk
over_cutoff = sum(1 for c in token_counts if c > CUTOFF)
print(f"{over_cutoff} of {len(token_counts)} summaries exceed {CUTOFF} tokens "
      f"({over_cutoff / len(token_counts):.1%})")


## Part 3 — Training

Build `prompt`/`completion` pairs (question + summary, answer separate —
required by the trainer to mask the loss to the answer only), quantize
the base model to 4-bit, attach LoRA adapters to the attention layers
(and MLP layers if `LITE_MODE=False`), and run `SFTTrainer`.


In [ ]:
for split in (train, val, test):
    for wine in split:
        wine.make_training_prompt(tokenizer, CUTOFF, do_round=True)

print(train[0].prompt)
print("---")
print(train[0].completion)


In [ ]:
import os
from datetime import datetime

import wandb
from datasets import Dataset, DatasetDict
from peft import LoraConfig
from transformers import set_seed
from trl import SFTConfig, SFTTrainer

# ── Constants ──────────────────────────────────────────────────────────────────
PROJECT_NAME: str = "wine-vfm"
HF_USER: str = "your-username"

LITE_MODE: bool = True  # True: free T4-friendly config. False: needs a paid A100.

RUN_NAME: str = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" + ("-lite" if LITE_MODE else "")
PROJECT_RUN_NAME: str = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME: str = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Overall hyperparameters
EPOCHS: int = 1 if LITE_MODE else 3
BATCH_SIZE: int = 16 if LITE_MODE else 128
MAX_SEQUENCE_LENGTH: int = CUTOFF + 20  # small margin for the prompt scaffolding
GRADIENT_ACCUMULATION_STEPS: int = 1

# QLoRA hyperparameters
LORA_R: int = 32 if LITE_MODE else 256
LORA_ALPHA: int = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT: float = 0.1

# Training hyperparameters
LEARNING_RATE: float = 1e-4
WARMUP_RATIO: float = 0.01
LR_SCHEDULER_TYPE: str = "cosine"
WEIGHT_DECAY: float = 0.001
OPTIMIZER: str = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8  # A100+ supports bf16 natively; T4 does not

VAL_SIZE: int = 200 if LITE_MODE else 1000
LOG_STEPS: int = 5 if LITE_MODE else 10
SAVE_STEPS: int = 50 if LITE_MODE else 200
LOG_TO_WANDB: bool = True


In [ ]:
# Optional: Weights & Biases tracking (add a WANDB_API_KEY Colab secret to use)
if LOG_TO_WANDB:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_API_KEY"] = wandb_api_key
    os.environ["WANDB_PROJECT"] = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"] = "false"
    wandb.login()
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)


In [ ]:
# ── Build the HF Dataset objects the trainer expects ───────────────────────────
train_ds = Dataset.from_list([w.to_datapoint() for w in train])
val_ds = Dataset.from_list([w.to_datapoint() for w in val[:VAL_SIZE]])
test_ds = Dataset.from_list([w.to_datapoint() for w in test])


In [ ]:
# ── Quantized base model — same 4-bit config validated in Part 1 ──────────────
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:,.1f} MB")


In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_parameters,
    args=train_parameters,
)


In [ ]:
# ── Train — pushes a checkpoint to the Hub every SAVE_STEPS, in case the ─────
# session disconnects. If it does, reload from HUB_MODEL_NAME + latest
# revision and resume (see PeftModel loading pattern in Part 4).
set_seed(42)
fine_tuning.train()

fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the Hub: {PROJECT_RUN_NAME}")

if LOG_TO_WANDB:
    wandb.finish()


## Part 4 — Testing

Load the frozen 4-bit base model plus the trained LoRA adapter from the
Hub, generate a VFM prediction for each test wine, and run it through the
SAME `Tester`/`evaluate` harness used for every other model in the ladder
— MAE ± 95% CI, R², scatter, error trend.


In [ ]:
from peft import PeftModel

from utils.evaluator import Tester, evaluate

REVISION: str | None = None  # set to a specific commit hash to pin a checkpoint

fine_tuned_model = (
    PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
    if REVISION else
    PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
)
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:,.1f} MB")


In [ ]:
def model_predict(wine) -> str:
    """Generate a VFM prediction from the fine-tuned adapter.

    Accepts a Wine object (matches the shared evaluator interface) rather
    than a raw dict — builds the inference prompt fresh so there is no
    dependency on train-time prompt/completion state.
    """
    prompt = f"{QUESTION}\n\n{wine.summary}\n\n{PREFIX}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


In [ ]:
from utils.items import PREFIX, QUESTION

set_seed(42)
evaluate(model_predict, test)


## Results Log

Copy this rung's MAE/CI/R² into the master Results Log alongside the
other models (random, constant, feature LR, BoW variants, XGBoost, DNN,
frontier zero-shot composite/direct) for the final comparison chart.
